# Czyszczenie i analiza zamówień e-commerce

Notebook pokazuje pełny proces pracy z brudnym zbiorem danych: od wygenerowania pliku CSV, przez eksplorację i czyszczenie, aż po analizę biznesową, wizualizację oraz zapis danych po obróbce.

## 1. Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## 2. Wygenerowanie danych wejściowych

Poniższa komórka tworzy plik `zamowienia_messy.csv`. W normalnym projekcie tę część uruchamia się tylko raz, ale w notebooku zostaje ona dodana po to, żeby całe rozwiązanie było samodzielne.

In [ ]:
np.random.seed(42)

n = 500
klienci = ["Anna Kowalska", "  Jan Nowak", "Anna Kowalska", "PIOTR WIŚNIEWSKI",
           "katarzyna lewandowska", "Tomasz Zieliński ", "Marta Wójcik",
           "anna kowalska ", "Krzysztof Kamiński", " Magdalena Dąbrowska"]
produkty = ["Laptop", "Mysz", "Klawiatura", "Monitor", "laptop", "MYSZ",
            "Słuchawki", "Pendrive", "monitor", "Webcam"]
kategorie = ["Elektronika", "elektronika", "ELEKTRONIKA", "Akcesoria",
             "akcesoria", "Akcesoria "]
miasta = ["Warszawa", "Kraków", "warszawa", "Gdańsk", "WROCŁAW",
          "Poznań", "Łódź ", " Warszawa", "kraków"]

start_date = datetime(2025, 1, 1)
daty_iso = [(start_date + timedelta(days=int(d))).strftime("%Y-%m-%d")
            for d in np.random.randint(0, 300, n // 2)]
daty_pl = [(start_date + timedelta(days=int(d))).strftime("%d.%m.%Y")
           for d in np.random.randint(0, 300, n // 2)]
daty = daty_iso + daty_pl
np.random.shuffle(daty)

df = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "klient": np.random.choice(klienci, n),
    "produkt": np.random.choice(produkty, n),
    "kategoria": np.random.choice(kategorie, n),
    "miasto": np.random.choice(miasta, n),
    "ilosc": np.random.choice([1, 2, 3, 5, -1, 0], n, p=[0.5, 0.2, 0.15, 0.1, 0.025, 0.025]),
    "cena_jednostkowa": np.random.choice(
        ["199.99", "299,99", "1 499.00", "89.50", "2999", "399.00 zł", None, "abc"],
        n
    ),
    "data_zamowienia": daty,
    "email": np.random.choice(
        ["anna@gmail.com", "JAN@WP.PL", "piotr.w@onet", "marta@gmail.com",
         "tomasz@interia.pl", None, "krzysztof.k@gmail.com", "brak"],
        n
    )
})

for col in ["miasto", "kategoria", "data_zamowienia"]:
    df.loc[df.sample(frac=0.05, random_state=1).index, col] = np.nan

df = pd.concat([df, df.sample(20, random_state=2)], ignore_index=True)

df.to_csv("zamowienia_messy.csv", index=False)
print(f"Wygenerowano plik 'zamowienia_messy.csv' — {len(df)} wierszy")

## 3. Wczytanie i eksploracja danych

In [ ]:
df = pd.read_csv("zamowienia_messy.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

In [ ]:
kolumny_kategoryczne = ["klient", "produkt", "kategoria", "miasto", "email"]

for kolumna in kolumny_kategoryczne:
    print(f"
--- {kolumna} ---")
    print(df[kolumna].value_counts(dropna=False))

## 4. Problemy jakościowe zauważone w danych

W danych można zauważyć kilka typowych problemów:

1. Występują zduplikowane rekordy.
2. Nazwy klientów, produktów, kategorii i miast są zapisane niejednolicie, np. z dodatkowymi spacjami albo różną wielkością liter.
3. Ceny mają różne formaty: kropki, przecinki, spacje, dopisek `zł`, wartości tekstowe i braki.
4. Daty są zapisane w dwóch formatach, co utrudnia automatyczną analizę czasu.
5. Część kolumn zawiera braki danych.
6. Kolumna `ilosc` zawiera wartości zerowe i ujemne, które nie powinny występować w poprawnych zamówieniach.
7. Niektóre adresy e-mail są niepoprawne albo wpisano tam wartość techniczną typu `brak`.

## 5. Czyszczenie danych

In [ ]:
df_clean = df.copy()

In [ ]:
# Usunięcie pełnych duplikatów wierszy
df_clean = df_clean.drop_duplicates()
df_clean.shape

In [ ]:
# Ujednolicenie tekstu: nazwy własne w title case, kategorie małymi literami
df_clean["klient"] = df_clean["klient"].astype("string").str.strip().str.lower().str.title()
df_clean["produkt"] = df_clean["produkt"].astype("string").str.strip().str.lower().str.title()
df_clean["miasto"] = df_clean["miasto"].astype("string").str.strip().str.lower().str.title()
df_clean["kategoria"] = df_clean["kategoria"].astype("string").str.strip().str.lower()

In [ ]:
# Konwersja dat zapisanych w dwóch formatach
df_clean["data_zamowienia"] = pd.to_datetime(
    df_clean["data_zamowienia"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [ ]:
# Oczyszczenie ceny i konwersja do typu float
df_clean["cena_jednostkowa"] = (
    df_clean["cena_jednostkowa"]
    .astype("string")
    .str.strip()
    .str.replace("zł", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df_clean["cena_jednostkowa"] = pd.to_numeric(df_clean["cena_jednostkowa"], errors="coerce")

In [ ]:
# Usunięcie rekordów bez kluczowych wartości
df_clean = df_clean.dropna(subset=["cena_jednostkowa", "data_zamowienia"])

# Uzupełnienie mniej krytycznych braków
df_clean["miasto"] = df_clean["miasto"].fillna("Unknown")
df_clean["kategoria"] = df_clean["kategoria"].fillna("unknown")
df_clean["email"] = df_clean["email"].fillna("brak_emaila")

In [ ]:
# Usunięcie błędnej liczby sztuk
df_clean = df_clean[df_clean["ilosc"] > 0].copy()

In [ ]:
df_clean.info()

In [ ]:
df_clean.isnull().sum()

## 6. Transformacje analityczne

In [ ]:
df_clean["wartosc_zamowienia"] = df_clean["ilosc"] * df_clean["cena_jednostkowa"]

In [ ]:
df_clean["rok"] = df_clean["data_zamowienia"].dt.year
df_clean["miesiac"] = df_clean["data_zamowienia"].dt.month
df_clean["nazwa_dnia"] = df_clean["data_zamowienia"].dt.day_name()

In [ ]:
df_clean["email_poprawny"] = df_clean["email"].astype(str).str.match(
    r"^[\w\.-]+@[\w\.-]+\.[A-Za-z]{2,}$"
)

df_clean.head()

## 7. Analiza biznesowa

In [ ]:
sprzedaz_miesieczna = (
    df_clean
    .groupby("miesiac", as_index=False)["wartosc_zamowienia"]
    .sum()
    .sort_values("miesiac")
)

sprzedaz_miesieczna

In [ ]:
top_klienci = (
    df_clean
    .groupby("klient", as_index=False)["wartosc_zamowienia"]
    .sum()
    .sort_values("wartosc_zamowienia", ascending=False)
    .head(5)
)

top_klienci

In [ ]:
srednia_kategoria = (
    df_clean
    .groupby("kategoria", as_index=False)["wartosc_zamowienia"]
    .mean()
    .sort_values("wartosc_zamowienia", ascending=False)
)

srednia_kategoria

## 8. Wizualizacja

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    sprzedaz_miesieczna["miesiac"],
    sprzedaz_miesieczna["wartosc_zamowienia"]
)

plt.title("Łączna wartość zamówień w poszczególnych miesiącach")
plt.xlabel("Miesiąc")
plt.ylabel("Wartość zamówień")
plt.xticks(sprzedaz_miesieczna["miesiac"])
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("wartosc_zamowien_miesiecznie.png", dpi=150)
plt.show()

## 9. Zapis oczyszczonego DataFrame

In [ ]:
df_clean.to_csv("zamowienia_clean.csv", index=False)
print("Zapisano oczyszczony plik: zamowienia_clean.csv")

## 10. Podsumowanie

Dane zostały przygotowane do analizy przez usunięcie duplikatów, standaryzację tekstu, konwersję dat i cen, obsługę braków oraz odrzucenie błędnych ilości. Po transformacjach możliwe było policzenie miesięcznej sprzedaży, wskazanie najlepszych klientów i porównanie średniej wartości zamówień między kategoriami.